# Orion en Google Colab (C++)

Este cuaderno **clona el repositorio**, **compila Orion en C++** y lo **ejecuta** en Colab con Ollama:
1. instala Ollama y descarga el modelo;
2. clona (o actualiza) `Miguel283332/Orion`;
3. compila con CMake y pasa las pruebas;
4. arranca Orion (con su guardián) y te deja hablar con él desde las celdas **y** desde su **interfaz web**.

> Las celdas son Python solo como *pegamento* de Colab: Orion es el programa en C++ que se compila aquí.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*. Sin GPU funciona, pero muy lento (usa entonces `qwen2.5:3b`).

## 1. Configuración

In [ ]:
REPO = "https://github.com/Miguel283332/Orion.git"
RAMA = "main"
MODELO = "qwen2.5:7b"      # con GPU T4 (16 GB). Sin GPU: "qwen2.5:3b"
USAR_DRIVE = True          # guardar su memoria en Google Drive (sobrevive a que se cierre Colab)
PUERTO_WEB = 7860

## 2. ¿Hay GPU?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "Sin GPU: irá lento. Cambia el entorno a GPU (T4) o usa MODELO = 'qwen2.5:3b'."

## 3. Instalar y arrancar Ollama, y descargar el modelo

In [ ]:
import subprocess, time, urllib.request, os

!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

def ollama_listo():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

if not ollama_listo():
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                                   env={**os.environ, "OLLAMA_KEEP_ALIVE": "60m"})
    for _ in range(60):
        if ollama_listo(): break
        time.sleep(1)
print("Ollama en marcha" if ollama_listo() else "Ollama NO arrancó: mira /content/ollama.log")

In [ ]:
!ollama pull {MODELO}

## 4. Clonar (o actualizar) el repositorio

In [ ]:
import os
if os.path.isdir("/content/Orion/.git"):
    !git -C /content/Orion fetch -q origin && git -C /content/Orion checkout -q {RAMA} && git -C /content/Orion pull -q origin {RAMA}
else:
    !git clone -q -b {RAMA} {REPO} /content/Orion
# para que Orion pueda hacer commit de sus propias mejoras dentro de este clon
!git -C /content/Orion config user.name "Orion (Colab)" && git -C /content/Orion config user.email "orion@localhost"
!git -C /content/Orion log --oneline -3

## 5. Dónde guarda su memoria

In [ ]:
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATOS = "/content/drive/MyDrive/orion_datos"
else:
    DATOS = "/content/orion_datos"
os.makedirs(DATOS, exist_ok=True)
print("Memoria de Orion en:", DATOS)

## 6. Compilar en C++ y pasar las pruebas

In [ ]:
%cd /content/Orion
!cmake --preset orion > /dev/null && cmake --build build -j"$(nproc)" 2>&1 | tail -n 3
!./build/orion --version
!./build/orion_tests | tail -n 3

## 7. Arrancar Orion

Se arranca **con su guardián** (la reinicia si se cae y aplica las versiones nuevas que compile de sí misma). Todo lo que dice queda en `/content/orion.log`.

In [ ]:
import subprocess, time, re

def arrancar_orion():
    global orion_proc, _leido
    if "orion_proc" in globals() and orion_proc.poll() is None:
        print("Orion ya está en marcha."); return
    _leido = 0
    orion_proc = subprocess.Popen(
        ["/content/Orion/build/guardian", "--datos", DATOS, "--modelo", MODELO,
         "--web-puerto", str(PUERTO_WEB), "--web-cualquier-host"],
        cwd="/content/Orion", stdin=subprocess.PIPE, stdout=open("/content/orion.log", "w"),
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    time.sleep(3)
    print(_nuevo())

def _nuevo():
    global _leido
    with open("/content/orion.log", encoding="utf-8", errors="replace") as f:
        f.seek(_leido)
        t = f.read()
        _leido = f.tell()
    return t

def orion(texto, max_s=900):
    """Habla con Orion (o manda un /comando) y muestra lo que responde."""
    if orion_proc.poll() is not None:
        print("Orion no está en marcha: ejecuta arrancar_orion()"); return
    _nuevo()                                  # descartar lo anterior
    orion_proc.stdin.write(texto + "\n"); orion_proc.stdin.flush()
    salida, inicio, ultimo = "", time.time(), time.time()
    es_comando = texto.strip().startswith("/")
    while time.time() - inicio < max_s:
        t = _nuevo()
        if t:
            print(t, end="", flush=True); salida += t; ultimo = time.time()
        if es_comando and salida and time.time() - ultimo > 1.5: break
        if not es_comando and (re.search(r"· \d+ tokens", salida) or "⚠" in salida or "interrumpida" in salida) and time.time() - ultimo > 1: break
        if orion_proc.poll() is not None: break
        time.sleep(0.3)

def actividad(n=30):
    """Lo último que ha hecho (también lo que piensa a solas)."""
    print("".join(open("/content/orion.log", encoding="utf-8", errors="replace").readlines()[-n:]))

arrancar_orion()

## 8. Interfaz web

Abre la interfaz de Orion (chat, actividad en directo, estado, metas, mejoras, acciones) en otra pestaña o aquí debajo.

In [ ]:
from google.colab import output
token = open(os.path.join(DATOS, "web_token.txt")).read().strip()
output.serve_kernel_port_as_window(PUERTO_WEB, path=f"/?t={token}")
# o dentro del cuaderno:
# output.serve_kernel_port_as_iframe(PUERTO_WEB, path=f"/?t={token}", height=700)

## 9. Hablar con Orion desde aquí

In [ ]:
orion("Hola, ¿quién eres y en qué hardware te estás ejecutando?")

In [ ]:
orion("/modelo")

In [ ]:
orion("/estado")

In [ ]:
# Algunas cosas que puedes probar:
# orion("busca en internet qué es la gravedad cuántica de bucles y guárdame un resumen en notas/lqg.txt")
# orion("/aprende cómo funciona git por dentro")
# orion("/practica ahora")
# orion("/mejorate")          # se mide, propone un cambio, lo mide y decide (tarda)
# orion("/mejoras")
# actividad()               # lo que ha hecho a solas
orion("/ayuda")

## 10. Apagar

`/salir` la apaga guardando todo (con Drive, su memoria sigue ahí la próxima vez).

> **Sus cambios de código** (si se mejora a sí misma) quedan como commits en **este clon** de Colab, que se borra al cerrar la sesión. Para conservarlos, súbelos a una rama: `!git -C /content/Orion push origin HEAD:orion-colab` (necesitas credenciales de GitHub), o descarga el repositorio.

In [ ]:
orion("/salir", max_s=30)

## Problemas frecuentes
- **`Ollama NO arrancó`**: vuelve a ejecutar la celda 3; mira `!tail /content/ollama.log`.
- **Muy lento**: sin GPU es normal. Cambia a GPU (T4) o usa `qwen2.5:3b`.
- **La celda de hablar no termina**: Orion puede tardar con respuestas largas; `orion("/para")` corta la respuesta.
- **Se cerró Colab**: vuelve a ejecutar todo; con `USAR_DRIVE = True` recuerda lo anterior.
- **Interfaz web en blanco**: vuelve a ejecutar la celda 8 (el enlace de Colab caduca).